In [ ]:
#| default_exp poll

# Poll

> Check data providers for new datasets, and for new versions of the datasets that handlers read

Each `RECORDS` entry of a handler that reads Zenodo data holds a record id in its `zenodo` field. A record is one version of a dataset. Zenodo never changes a published record, so the record id fixes the exact files the handler reads. Providers publish corrections and extensions as new records, under the same parent id. This module finds the records that handlers read and that have a newer version, and the datasets in a Zenodo community that no handler reads.

In [ ]:
#| export
import pandas as pd
from importlib import import_module
from marisco.cli.ingest import handlers
from marisco.invenio import fetch_latest, community_records
from marisco.configs import ZENODO_COMMUNITY

In [ ]:
from fastcore.test import *

## Records that handlers read

In [ ]:
#| export
def handler_records() -> pd.DataFrame: # One row per record: `handler`, `key` (the `RECORDS` key) and `record_id`
    "Zenodo records that the handlers' `RECORDS` entries read"
    rows = [(h, key, rec['zenodo']) for h in handlers()
        for key, rec in getattr(import_module(f'marisco.handlers.{h}'), 'RECORDS', {}).items() if 'zenodo' in rec]
    return pd.DataFrame(rows, columns=['handler', 'key', 'record_id'])

`handler_records` reads the record ids without calling Zenodo. `fram_strait` and `jois` read ten records:

In [ ]:
recs = handler_records()
test_eq(recs.groupby('handler').size().to_dict(), {'fram_strait': 6, 'jois': 4})
recs

In [ ]:
#| export
def check_records(
    recs: pd.DataFrame # Records, as `handler_records` returns them
) -> pd.DataFrame: # `recs` with the dataset's `parent_id`, its `latest_id`, and whether the record is `current`
    "Compare each record that a handler reads with the latest version of its dataset on Zenodo"
    latest = [fetch_latest(r) for r in recs.record_id]
    return recs.assign(parent_id=[r['parent']['id'] for r in latest], latest_id=[int(r['id']) for r in latest],
        current=lambda d: d.record_id == d.latest_id)

`check_records` makes one request for each record. `FS2024_i129` reads record 20761705, but its dataset has a newer version, record 23111574, which adds U-236. Moving `fram_strait` to the newer record is a curation decision for the handler:

In [ ]:
#| eval: false
checked = check_records(recs)
test_eq(checked.loc[~checked.current, 'key'].tolist(), ['FS2024_i129'])
checked

## New datasets

In [ ]:
#| export
def new_datasets(
    checked: pd.DataFrame, # Records, as `check_records` returns them
    slug: str=ZENODO_COMMUNITY # Zenodo community
) -> pd.DataFrame: # One row per dataset: `record_id` of its latest version, `parent_id`, `created` and `title`
    "Datasets in community `slug` that no handler reads"
    parents = set(checked.parent_id)
    rows = [(int(r['id']), r['parent']['id'], r['created'][:10], r['metadata']['title'])
        for r in community_records(slug) if r['parent']['id'] not in parents]
    return pd.DataFrame(rows, columns=['record_id', 'parent_id', 'created', 'title'])

`new_datasets` compares datasets by parent id, not by record id. The community lists only the latest version of each dataset. The newer Fram Strait 2024 record therefore belongs to a dataset that `fram_strait` reads, through `FS2024_i129`, and is not a new dataset. The remaining records are datasets that no handler reads:

In [ ]:
#| eval: false
new = new_datasets(checked)
test_eq(23111574 in new.record_id.values, False)
new